In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# מבוא לפרימיטיבים

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** גרסת הבטא של מודל הרצה חדש זמינה כעת. מודל ה-Directed execution מספק גמישות רבה יותר בהתאמה אישית של זרימת העבודה להפחתת שגיאות. עיין במדריך [Directed execution model](/guides/directed-execution-model) לקבלת מידע נוסף.

<span id="qpu-access-patterns"></span>
## מדוע Qiskit הציגה פרימיטיבים?
בדומה לימים הראשונים של המחשבים הקלאסיים, כאשר מפתחים היו צריכים לתפעל רגיסטרים של ה-CPU ישירות, הממשק המוקדם ל-QPUs החזיר פשוט את הנתונים הגולמיים מהאלקטרוניקה השולטת.
זה לא היה בעיה גדולה כאשר QPUs שכנו במעבדות והרשו גישה ישירה רק לחוקרים.
מתוך הכרה בכך שרוב המפתחים לא יכירו ולא צריכים להכיר את אופן הפענוח של נתונים גולמיים כאלה ל-0 ו-1, Qiskit הציגה את `backend.run`, הפשטה ראשונה לגישה ל-QPUs בענן. זה אפשר למפתחים לעבוד עם פורמט נתונים מוכר ולהתמקד בתמונה הגדולה.

כאשר הגישה ל-QPUs הפכה נפוצה יותר, ועם יותר אלגוריתמים קוונטיים שפותחו, עלתה שוב הצורך בהפשטה ברמה גבוהה יותר. בתגובה, Qiskit הציגה את ממשק הפרימיטיבים, המותאם לשתי משימות ליבה בפיתוח אלגוריתמים קוונטיים: אמדון ערך ציפייה (`Estimator`) ודגימת Circuit (`Sampler`). המטרה היא שוב לעזור למפתחים להתמקד יותר בחדשנות ופחות בהמרת נתונים. ממשק הפרימיטיבים מחליף את ממשק `backend.run`, שכן `Sampler` מספק את אותה גישה ישירה לחומרה שהוצעה על ידי `backend.run`.

## מהו פרימיטיב?
מערכות מחשוב בנויות על שכבות מרובות של הפשטה. הפשטות מאפשרות לך להתמקד ברמת פרטים מסוימת הרלוונטית למשימה שבידיך. ככל שמתקרבים לחומרה, כך נמוכה יותר רמת ההפשטה הנדרשת (לדוגמה, ייתכן שתצטרך להזיז או לתפעל נתונים ברמת הוראות ה-CPU). ככל שהמשימה שברצונך לבצע מורכבת יותר, כך ההפשטות יהיו ברמה גבוהה יותר (לדוגמה, אפשר שתשתמש בספרייה לביצוע חישובים אלגבריים).

בהקשר זה, *פרימיטיב* הוא הוראת העיבוד הקטנה ביותר, אבן הבניין הפשוטה ביותר ממנה ניתן ליצור משהו שימושי לרמת הפשטה נתונה.

ההתקדמות האחרונה בחישוב קוונטי הגבירה את הצורך לעבוד ברמות גבוהות יותר של הפשטה. ככל שהתחום מתקדם לעבר יחידות עיבוד קוונטי (QPUs) גדולות יותר וזרימות עבודה מורכבות יותר, המיקוד עובר מאינטראקציה עם אותות qubit בודדים לראיית המכשירים הקוונטיים כמערכות המבצעות משימות הכרחיות.

שתי המשימות הנפוצות ביותר למחשבים קוונטיים הן דגימת מצבים קוונטיים וחישוב ערכי ציפייה. משימות אלה הניעו את עיצוב פרימיטיבי Qiskit: **Estimator** ו-**Sampler**.

- Estimator מחשב ערכי ציפייה של אובזרבבלים ביחס למצבים המוכנים על ידי Circuits קוונטיות.
- Sampler דוגם את רגיסטר הפלט מהרצת Circuit קוונטית.

בקצרה, מודל החישוב שהוצג על ידי פרימיטיבי Qiskit מקדם את התכנות הקוונטי צעד אחד קרוב יותר למקום שבו נמצא התכנות הקלאסי כיום, שבו המיקוד פחות על פרטי החומרה ויותר על התוצאות שאתה מנסה להשיג.

## הגדרת פרימיטיב ומימושים
ישנם שני סוגים של פרימיטיבי Qiskit: המחלקות הבסיסיות והמימושים שלהן. פרימיטיבי Qiskit מוגדרים על ידי מחלקות בסיס פרימיטיב בקוד פתוח שחיות ב-Qiskit SDK (במודול [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives)). ספקים (כגון Qiskit Runtime) יכולים להשתמש במחלקות בסיס אלה כדי לגזור מימושי Sampler ו-Estimator משלהם. רוב המשתמשים יתקשרו עם מימושי הספקים, לא עם הפרימיטיבים הבסיסיים.

### מחלקות בסיס
[`BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) ו-[`BaseSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV2) — מחלקות בסיס מופשטות המגדירות ממשק משותף למימוש פרימיטיבים. כל שאר המחלקות במודול [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) יורשות ממחלקות בסיס אלה. מפתחים צריכים להשתמש בהן אם הם מעוניינים ליצור מודל הרצה משלהם מבוסס פרימיטיבים עבור ספק ספציפי. מחלקות אלה עשויות להיות שימושיות גם למי שרוצה לבצע עיבוד מותאם מאוד ומגלה שמימושי הפרימיטיבים הקיימים פשוטים מדי לצרכיו. משתמשים כלליים לא ישתמשו ישירות במחלקות הבסיס.

<span id="implementations"></span>
### מימושים
אלה הם מימושים של מחלקות הבסיס של הפרימיטיבים:

- פרימיטיבי Qiskit Runtime ([`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) ו-[`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2)) מספקים מימוש מתוחכם יותר (לדוגמה, על ידי הכללת הפחתת שגיאות) כשירות מבוסס ענן. מימוש זה של הפרימיטיבים הבסיסיים משמש לגישה לחומרה של IBM Quantum&reg;. הם נגישים דרך IBM Qiskit Runtime.

- [`StatevectorEstimator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorEstimator) ו-[`StatevectorSampler`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) — מימושי ייחוס של הפרימיטיבים המשתמשים בסימולטור המובנה ב-Qiskit. הם בנויים עם מודול [`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info#quantum-information) של Qiskit, ומייצרים תוצאות המבוססות על סימולציות וקטור מצב אידיאליות. הם נגישים דרך Qiskit.

- [`BackendEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) ו-[`BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) — ניתן להשתמש במחלקות אלה כדי "לעטוף" כל משאב חישוב קוונטי בפרימיטיב. זה מאפשר לך לכתוב קוד בסגנון פרימיטיב עבור ספקים שעדיין אין להם ממשק מבוסס פרימיטיבים. ניתן להשתמש במחלקות אלה בדיוק כמו Sampler ו-Estimator הרגילים, אלא שיש לאתחלן עם ארגומנט `backend` נוסף לבחירת מחשב הקוונטי להרצה עליו. הם נגישים באמצעות Qiskit.

## יתרונות פרימיטיבי Qiskit
עם פרימיטיבים, משתמשי Qiskit יכולים לכתוב קוד קוונטי עבור QPU ספציפי מבלי לנהל כל פרט באופן מפורש. כמו כן, בשל שכבת ההפשטה הנוספת, ייתכן שתוכל לגשת ביתר קלות ליכולות החומרה המתקדמות של ספק נתון. לדוגמה, עם פרימיטיבי Qiskit Runtime, תוכל לנצל את ההתקדמות האחרונה בהפחתה ודיכוי שגיאות על ידי שינוי אפשרויות כגון [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level) של הפרימיטיב, במקום לבנות מימוש משלך של טכניקות אלה.

עבור ספקי חומרה, מימוש פרימיטיבים באופן מקורי מאפשר לך לספק למשתמשים שלך דרך "מהקופסה" יותר לגשת לתכונות החומרה שלך כגון טכניקות עיבוד לאחר מתקדמות. לכן קל יותר למשתמשים שלך ליהנות מהיכולות הטובות ביותר של החומרה שלך.

## פרטי פרימיטיב
כפי שתואר קודם, כל הפרימיטיבים נוצרים ממחלקות הבסיס; לפיכך, יש להם אותה מבנה ושימוש כלליים. לדוגמה, פורמט הקלט עבור כל פרימיטיבי Estimator זהה. עם זאת, ישנם הבדלים במימושים שהופכים אותם לייחודיים.

> **Note:** מכיוון שרוב המשתמשים ניגשים לפרימיטיבי Qiskit Runtime, הדוגמאות בשאר סעיף זה מבוססות על פרימיטיבי Qiskit Runtime.

<span id="estimator"></span>
### Estimator
פרימיטיב Estimator מחשב ערכי ציפייה עבור אובזרבבל אחד או יותר ביחס למצבים המוכנים על ידי Circuits קוונטיות. ה-Circuits יכולות להיות בעלות פרמטרים, כל עוד ערכי הפרמטרים ניתנים גם הם כקלט לפרימיטיב.

הקלט הוא מערך של [PUBs.](/guides/primitive-input-output#pubs) כל PUB הוא בפורמט:

(`<single circuit>`, `<one or more observables>`, `<optional one or more parameter values>`, `<optional precision>`),

כאשר `parameter values` האופציונלי יכול להיות רשימה או פרמטר יחיד. מימושי Estimator שונים תומכים באפשרויות תצורה שונות. אם הקלט מכיל מדידות, הן מתעלמות.

הפלט הוא [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult) המכיל את ערכי הציפייה המחושבים לכל זוג, ושגיאות הסטנדרטיות שלהם, בצורת `PubResult`. כל `PubResult` מכיל גם נתונים וגם מטא-נתונים.

ה-Estimator משלב אלמנטים מאובזרבבלים וערכי פרמטרים בעקבות כללי שידור NumPy כמתואר בנושא [קלט ופלט של פרימיטיבים](primitive-input-output#broadcasting-rules).

דוגמה:

In [1]:
# This cell is hidden from users, it creates the circuits and observables to run

from qiskit_ibm_runtime import EstimatorV2, SamplerV2, QiskitRuntimeService
from qiskit.circuit.random import random_circuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
import numpy as np

service = QiskitRuntimeService()
backend = service.least_busy()
phi = Parameter("phi")

circuit1 = random_circuit(10, 5, seed=12345)
circuit1.rzz(phi, 1, 2)
observable1 = SparsePauliOp.from_sparse_list(
    [("ZXYZ", [1, 2, 3, 4], 1)], num_qubits=10
)
param_values1 = np.random.uniform(size=5).T

circuit2 = random_circuit(10, 5, seed=12345)
circuit2.rzz(phi, 1, 2)
observable2 = SparsePauliOp.from_sparse_list(
    [("XZYX", [1, 2, 3, 4], 1)], num_qubits=10
)
param_values2 = np.random.uniform(size=5).T

shots1 = 164
shots2 = 1024

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
circuit1 = pm.run(circuit1)
circuit2 = pm.run(circuit2)
observable1 = observable1.apply_layout(circuit1.layout)
observable2 = observable2.apply_layout(circuit2.layout)

In [2]:
estimator = EstimatorV2(mode=backend)
estimator_job = estimator.run(
    [
        (circuit1, observable1, param_values1),
        (circuit2, observable2, param_values2),
    ]
)

<span id="sampler"></span>
### Sampler
המשימה המרכזית של Sampler היא דגימת רגיסטר הפלט מהרצה של Circuit קוונטית אחת או יותר. ה-Circuits יכולות להיות בעלות פרמטרים, כל עוד ערכי הפרמטרים ניתנים גם הם כקלט לפרימיטיב.

הקלט הוא [PUBs](/guides/primitive-input-output#pubs) אחד או יותר, בפורמט:

(`<single circuit>`, `<one or more optional parameter value>`, `<optional shots>`),

כאשר יכולים להיות מספר פריטי `parameter values`, וכל פריט יכול להיות מערך או פרמטר יחיד, תלוי ב-Circuit שנבחר. בנוסף, הקלט חייב להכיל מדידות.

הפלט הוא ספירות או מדידות לכל shot, כאובייקטים של [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult), ללא משקלים. עם זאת, למחלקת התוצאות יש מתודות להחזרת דגימות משוקללות, כגון ספירות. עיין ב[קלט ופלט של פרימיטיבים](primitive-input-output#broadcasting-rules) לפרטים מלאים.

דוגמה:

In [3]:
# This cell is hidden from users, add measurement instructions to circuits
circuit1.measure_active()
circuit2.measure_active()

In [4]:
sampler = SamplerV2(mode=backend)
sampler_job = sampler.run(
    [
        (circuit1, param_values1, shots1),
        (circuit2, param_values2, shots2),
    ]
)

## הצעדים הבאים
> **Tip:** - קרא [תחילת העבודה עם פרימיטיבים](get-started-with-primitives) כדי ליישם פרימיטיבים בעבודתך.
>     - עיין ב[דוגמאות מפורטות של פרימיטיבים.](primitives-examples)
>     - תרגל עם פרימיטיבים על ידי עבודה דרך [שיעור פונקציית עלות](/learning/courses/variational-algorithm-design/cost-functions) ב-IBM Quantum Learning.
>     - עיין ב[עזר API של EstimatorV2](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) וב[עזר API של SamplerV2](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2).
>     - קרא [העברה לפרימיטיבים V2](/guides/v2-primitives).